In [1]:
import numpy as np
from PIL import Image
import cv2
import torch
import matplotlib.pyplot as plt
from transformers import pipeline
from PIL import Image
import requests

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
image_url = './Test_IMG.jpg'

# Camera parameters
FOCAL_LENGTH = 50.0      # Focal length in mm
F_NUMBER = 2.8          # Aperture f
SENSOR_WIDTH = 36.0      # Full-frame sensor width in mm
MAX_BLUR_PX = 40         # Maximum blur radius

print("\n=== Parameter Adjustment Tips ===")
print(f"Current settings:")
print(f"  Focal length: {FOCAL_LENGTH}mm")
print(f"  Aperture: f/{F_NUMBER}")
# print(f"  Focus distance: {FOCUS_DISTANCE}m")


=== Parameter Adjustment Tips ===
Current settings:
  Focal length: 50.0mm
  Aperture: f/2.8


In [6]:

#1.load model and process image
pipe = pipeline(task="depth-estimation", model="LiheYoung/depth-anything-large-hf")

#Load original image
I_full = np.asarray(Image.open(image_url).convert("RGB"), dtype=np.float32) / 255.0
H0, W0 = I_full.shape[:2]
print(f"Image size: {W0}x{H0}")

# Load and process depth map
# Z_t = torch.load("./depth_tensor.pt", map_location="cpu")
Z_t = torch.tensor(np.array(pipe(Image.open(image_url))["depth"]))
# print(Z_t)
if isinstance(Z_t, dict) and "image_depth" in Z_t:
    Z_t = Z_t["image_depth"]

Z = Z_t.detach().cpu().numpy().astype(np.float32)
# Z = Z_t.detach().cpu().astype(np.float32)
print(f"Original depth range: [{Z.min():.3f}, {Z.max():.3f}]")
Z = (Z - Z.min()) / (Z.max() - Z.min() + 1e-8)
# Resize the depth's dimension to match the original one because the depth model can result in smaller dimension depth map
Z = cv2.resize(Z, (W0, H0), interpolation=cv2.INTER_LINEAR)
Z = cv2.bilateralFilter(Z.astype(np.float32), d=9, sigmaColor=0.05, sigmaSpace=5)

# Map depth to actual distance (adjust based on scene)
# Use exponential mapping for more realistic depth distribution
Z_min, Z_max = 0.5, 100.0  # meters
## THE WAY WE MAP DEPTH VALUES TO REAL DISTANCES CAN SIGNIFICANTLY AFFECT THE DOF EFFECT
Z_real = Z_min * np.exp(Z * np.log(Z_max / Z_min))
print(f"Mapped depth range: [{Z_real.min():.2f}m, {Z_real.max():.2f}m]")

# Clean up outliers
Z_real = np.clip(Z_real, Z_min, Z_max)
Z_real = cv2.medianBlur(Z_real, 3)

# 2.Camera parameters and CoC calculation based on the formula
##C oC = (A * |s - z| * f) / (z * (s - f))
def compute_coc_radius(Z_m, focal_length_mm, f_number, focus_distance_m, 
                       sensor_width_mm, image_width_px, max_blur_px=40):
    """
    Calculate Circle of Confusion radius in pixels
    
    Parameters:
    - Z_m: Depth map
    - focal_length_mm: Focal length
    - f_number: Aperture f-number
    - focus_distance_m: Focus distance
    - sensor_width_mm: Sensor width
    - image_width_px: Image width
    - max_blur_px: Maximum blur radius
    """
    pixel_size_mm = sensor_width_mm / image_width_px
    
    f_m = focal_length_mm / 1000.0
    s = focus_distance_m
    
    aperture_diameter_mm = focal_length_mm / f_number
    
    # CoC = (A * |s - z| * f) / (z * (s - f))
    eps = 1e-6
    numerator = aperture_diameter_mm * np.abs(s - Z_m) * f_m
    denominator = Z_m * (s - f_m) + eps
    coc_diameter_mm = numerator / (denominator + eps)
    
    coc_radius_px = (coc_diameter_mm / pixel_size_mm) / 2.0

    coc_radius_px = np.clip(coc_radius_px, 0.0, max_blur_px)
    coc_radius_px = np.nan_to_num(coc_radius_px, nan=0.0, posinf=max_blur_px)
    
    return coc_radius_px.astype(np.float32)


# Manully select the focus
def select_focus_interactive(image, depth_map):
    selected_point = {'x': W0//2, 'y': H0//2, 'depth': None}
    display = image.copy()
    
    def mouse_callback(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            selected_point['x'] = x
            selected_point['y'] = y
            selected_point['depth'] = depth_map[y, x]
            
            display_temp = image.copy()
            cv2.drawMarker(display_temp, (x, y), (0, 1, 0), 
                          cv2.MARKER_CROSS, 30, 2)
            info_text = f"Focus: ({x}, {y}) Depth: {selected_point['depth']:.2f}m"
            cv2.putText(display_temp, info_text, (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 1, 0), 2)
            
            cv2.imshow('Select Focus Point', 
                      cv2.cvtColor((display_temp * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
            print(f"✓ Selected point: ({x}, {y}), Depth: {selected_point['depth']:.2f}m")
    
    cv2.namedWindow('Select Focus Point', cv2.WINDOW_NORMAL)
    cv2.setMouseCallback('Select Focus Point', mouse_callback)
    
    display_with_text = display.copy()
    cv2.putText(display_with_text, "Click on object to focus, press Q to confirm", 
               (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (1, 1, 0), 2)
    
    cv2.imshow('Select Focus Point', 
              cv2.cvtColor((display_with_text * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
    
    print("1. Click on the object you want to focus on")
    print("2. Press 'q' or ESC to confirm selection\n")
    
    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:  # q or ESC
            break
    
    cv2.destroyAllWindows()
    
    if selected_point['depth'] is None:
        selected_point['depth'] = depth_map[selected_point['y'], selected_point['x']]
    
    return selected_point['depth']

# Use interactive selection (comment out to use manual setting)
FOCUS_DISTANCE = select_focus_interactive(I_full, Z_real)
print(f"\n>>> Final focus distance: {FOCUS_DISTANCE:.2f}m\n")

# Calculate CoC
coc_radius = compute_coc_radius(
    Z_real, FOCAL_LENGTH, F_NUMBER, FOCUS_DISTANCE,
    SENSOR_WIDTH, W0, MAX_BLUR_PX
)

## 5. Layered blur
def layered_blur(image, coc_map, num_layers=40, max_radius=40):
    """
    Variable blur effect using layered blur
    
    Parameters:
    - image: Input image [H,W,3]
    - coc_map: CoC radius map [H,W]
    - num_layers: Number of blur layers
    - max_radius: Maximum blur radius
    """
    H, W = image.shape[:2]
    
    # Create blur layers
    blur_radii = np.linspace(0, max_radius, num_layers)
    blur_layers = []
    
    print("Generating blur layers...")
    for i, radius in enumerate(blur_radii):
        if radius < 0.5:
            blur_layers.append(image.copy())
        else:
            # Use Gaussian blur (faster than disk convolution)
            sigma = radius / 2.0  # Empirical conversion
            blurred = cv2.GaussianBlur(image, (0, 0), sigma, borderType=cv2.BORDER_REFLECT)
            blur_layers.append(blurred)
        print(f"  Layer {i+1}/{num_layers}: radius={radius:.1f}px", end='\r')
    print()
    
    # Select appropriate blur layer for each pixel (vectorized)
    # Map CoC to layer indices
    layer_indices = np.clip(
        (coc_map / max_radius) * (num_layers - 1),
        0, num_layers - 1
    )
    
    idx_low = np.floor(layer_indices).astype(np.int32)
    idx_high = np.ceil(layer_indices).astype(np.int32)
    alpha = (layer_indices - idx_low).astype(np.float32)
    
    # Interpolate blend
    print("Blending layers...")
    result = np.zeros_like(image)
    for i in range(num_layers):
        mask_low = (idx_low == i).astype(np.float32)
        mask_high = (idx_high == i).astype(np.float32)
        
        weight_low = mask_low * (1 - alpha)
        weight_high = mask_high * alpha
        weight_total = weight_low + weight_high
        
        if weight_total.max() > 0:
            result += blur_layers[i] * weight_total[:, :, np.newaxis]
    
    return np.clip(result, 0.0, 1.0)

# Execute layered blur
print("\nStarting depth-of-field simulation...")
result = layered_blur(I_full, coc_radius, num_layers=40, max_radius=MAX_BLUR_PX)

## Save results
# Save blur result
output_path = "refocus_optimized.png"
cv2.imwrite(output_path, cv2.cvtColor((result * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
print(f"Saved: {output_path}")

# Save depth visualization
depth_vis = ((Z_real - Z_real.min()) / (Z_real.max() - Z_real.min()) * 255).astype(np.uint8)
depth_vis = cv2.applyColorMap(depth_vis, cv2.COLORMAP_TURBO)
cv2.imwrite("depth_visualization.png", depth_vis)
print("Saved depth visualization: depth_visualization.png")

# Save CoC visualization
coc_vis = (coc_radius / MAX_BLUR_PX * 255).astype(np.uint8)
coc_vis = cv2.applyColorMap(coc_vis, cv2.COLORMAP_JET)
cv2.imwrite("coc_visualization.png", coc_vis)
print("Saved CoC visualization: coc_visualization.png")


print(f"\nInteresting adjustments guild:")
print(f"  - Increase aperture (decrease F_NUMBER): Stronger depth-of-field effect")
print(f"  - Increase focal length: Stronger depth-of-field effect")
print(f"  - Adjust FOCUS_DISTANCE to the depth you want sharp")
print(f"  - Modify Z_min/Z_max to match actual scene depth range")


Device set to use mps:0


Image size: 4766x4760
Original depth range: [0.000, 255.000]
Mapped depth range: [0.53m, 97.49m]
1. Click on the object you want to focus on
2. Press 'q' or ESC to confirm selection

✓ Selected point: (2717, 3524), Depth: 4.68m

>>> Final focus distance: 4.68m


Starting depth-of-field simulation...
Generating blur layers...
  Layer 40/40: radius=40.0px
Blending layers...
Saved: refocus_optimized.png
Saved depth visualization: depth_visualization.png
Saved CoC visualization: coc_visualization.png

Interesting adjustments guild:
  - Increase aperture (decrease F_NUMBER): Stronger depth-of-field effect
  - Increase focal length: Stronger depth-of-field effect
  - Adjust FOCUS_DISTANCE to the depth you want sharp
  - Modify Z_min/Z_max to match actual scene depth range


: 